# Session 3 — Signal Region and Analysis Strategy

Here we combine the object selections from Session 2 into a **full signal region (SR)** for the bb + MET search, closely mirroring the strategy used in the CMS JHEP 02 (2025) 050 analysis. We will also look at how the main backgrounds populate the SR and how control regions are used conceptually.

**Learning objectives:**
- Define and implement the bb + MET signal region for the bbDM search
- Apply full event selection and study the resulting MET distribution
- Compare event yields before and after cuts (cutflow and yield tables)
- Understand background composition and the role of control regions

**Duration:** ~3 hours

![Placeholder: cartoon of SR and CR definition in the bb + MET analysis](figures/session3_fig_sr_cr_cartoon.png)

---
## 1. Signal Region Definition

We define the **signal region (SR)** as:

- **MET > 200 GeV**
- **≥ 2 b-jets** (using the medium b-tag WP)
- **0 isolated leptons** (lepton veto)

This targets the signature pp → b b̄ + χ χ̄ while suppressing many backgrounds.

---
## 2. Backgrounds in the Signal Region

| Background | Why it enters | How we suppress |
|------------|----------------|------------------|
| **tt̄** | Has b-jets; semileptonic has MET from ν | Lepton veto removes most; MET cut reduces rest |
| **Z → νν + jets** | Invisible νν → large MET | Dominant at high MET; need MC/data-driven estimate |
| **W + jets** | W → ℓν gives MET | Lepton veto removes W→ℓν |
| **QCD** | Mismeasured jets → fake MET | MET and b-jet requirements reduce it |

**Discussion:** Why does MET > 200 GeV suppress W+jets more than Z→νν?

In [ ]:
# Ensure project root is on sys.path (for SWAN / any kernel)
import sys, os
if 'config' not in sys.modules:
    root = os.path.abspath(os.getcwd())
    while root != os.path.dirname(root):
        if os.path.isdir(os.path.join(root, 'config')) and os.path.isfile(os.path.join(root, 'requirements.txt')):
            sys.path.insert(0, root)
            break
        root = os.path.dirname(root)

import numpy as np
import awkward as ak
import matplotlib.pyplot as plt

# Re-use selection from Session 2 (or import from processor)
MET_SR_MIN = 200.0  # GeV

# Assume we have: good_jets, nbjets, nlep, met from Session 2
# Signal region mask:
# njets = ak.num(good_jets)
# sr_mask = (njets >= 1) & (nbjets >= 2) & (nlep == 0) & (met > MET_SR_MIN)

---
## 3. Full Event Selection

Combine all steps:
1. Good jets (pT, η, jetId)
2. Count b-jets (DeepFlavB > 0.2783)
3. Count tight leptons (veto: require 0)
4. MET > 200 GeV
5. Require ≥ 2 b-jets

Implement this and count events passing each step (cutflow).

In [ ]:
# Paste or import your Session 2 selection functions, then:

# good_jets = select_good_jets(events)
# njets = ak.num(good_jets)
# nbjets = count_bjets(good_jets)
# nlep = n_tight_leptons(events)
# met = events.MET.pt

# Cutflow
# N0 = len(events)
# N1 = ak.sum(njets >= 1)
# N2 = ak.sum((njets >= 1) & (nbjets >= 2))
# N3 = ak.sum((njets >= 1) & (nbjets >= 2) & (nlep == 0))
# N4 = ak.sum((njets >= 1) & (nbjets >= 2) & (nlep == 0) & (met > MET_SR_MIN))
# print("After ≥1 jet:", N1)
# print("After ≥2 b-jets:", N2)
# print("After lepton veto:", N3)
# print("After MET > 200 GeV (SR):", N4)

### Exercise 3.1 — Cutflow
Implement the cutflow above and fill a table: **Cut** | **Yield**. Use your NanoAOD sample (one process is enough for the exercise).

In [ ]:
# Your code: cutflow table

---
## 4. MET Distribution in the Signal Region

Plot **MET** for events that pass the full signal region (MET > 200 GeV, ≥2 b-jets, 0 leptons). Use bins starting at 200 GeV (e.g. 200–600 GeV in 50 GeV bins).

In [ ]:
# sr_mask = (njets >= 1) & (nbjets >= 2) & (nlep == 0) & (met > MET_SR_MIN)
# met_sr = met[sr_mask]
# plt.hist(ak.to_numpy(met_sr), bins=50, range=(200, 700), edgecolor="black", alpha=0.7)
# plt.xlabel("MET [GeV]")
# plt.ylabel("Events")
# plt.title("MET in signal region")
# plt.show()

### Exercise 3.2 — MET after selection
Produce the MET distribution in the signal region. Add axis labels and a title. If you have multiple samples (e.g. tt̄, Z→νν), you can plot them stacked or overlaid.

In [ ]:
# Your code: MET in SR

---
## 5. Yields Before and After Cuts

Build a **yield table** with one row per cut and one column per process (if you have multiple samples). Example:

| Cut | tt̄ | Z→νν | W+jets |
|-----|-----|------|--------|
| Preselection | ... | ... | ... |
| ≥2 b-jets | ... | ... | ... |
| 0 leptons | ... | ... | ... |
| MET > 200 GeV | ... | ... | ... |

In [ ]:
# Example for a single sample:
# import pandas as pd
# df = pd.DataFrame({
#     "Cut": ["Presel", "≥2 b-jets", "0 leptons", "MET>200"],
#     "Yield": [N1, N2, N3, N4]
# })
# print(df)

### Exercise 3.3 — Yield table
Create a yield table for your sample(s). Discuss: which background survives most in the SR? Why?

In [ ]:
# Your code: yield table

---
## 6. Control Regions

**Control regions** are used to validate background modelling with data. Example:
- **Single-lepton region:** require exactly 1 tight lepton, ≥2 b-jets, MET > 50 GeV. Dominated by tt̄; we can check that MC describes data before relying on it in the SR.

We do not implement CRs in full here, but the idea is: measure backgrounds in CRs and transfer to the SR (with transfer factors or fits).

### Discussion
What physics process produces large MET in Z→νν? Why is Z→νν a major background in our signal region?

---
## 7. Using the Coffea Processor

The provided **bbdm_processor.py** does the full selection and fills histograms. You can run it on one or more files to get merged results.

Example (pseudo-code):
```python
from processor.bbdm_processor import bbDMProcessor
from coffea.processor import run_uproot_job
files = {"ttbar": ["file1.root"]}
out = run_uproot_job(files, "events", bbDMProcessor(), executor=iterative_executor)
```
Then inspect `out["met_sr"]`, `out["cutflow"]`, etc.

In [ ]:
# Optional: run the processor (uncomment and set paths)
# import sys
# sys.path.insert(0, "processor")
# from bbdm_processor import bbDMProcessor, get_nanoevents
# proc = bbDMProcessor()
# events = get_nanoevents("path/to/file.root")
# result = proc.process(events)
# print("Cutflow:", result["cutflow"])
# result["met_sr"].plot()

---
## 8. Comparison Plots

Produce **comparison plots**: e.g. MET distribution **before** any SR cuts (but after ≥1 jet) and **after** full SR selection. This shows how the cut shapes the distribution.

In [ ]:
# Example: MET before and after SR
# met_presel = met[njets >= 1]
# met_sr = met[sr_mask]
# fig, ax = plt.subplots(1, 2, figsize=(10, 4))
# ax[0].hist(ak.to_numpy(met_presel), bins=50, range=(0, 500), label="Presel", alpha=0.7)
# ax[1].hist(ak.to_numpy(met_sr), bins=50, range=(200, 600), label="SR", alpha=0.7)
# ax[0].set_xlabel("MET [GeV]"); ax[1].set_xlabel("MET [GeV]")
# plt.legend(); plt.tight_layout(); plt.show()

### Exercise 3.4
Make a comparison: MET after preselection (≥1 jet) vs MET in signal region. Use two side-by-side or overlaid histograms with legends.

In [ ]:
# Your code: comparison plot

---
## 9. Summary — Session 3

- **Signal region:** MET > 200 GeV, ≥2 b-jets, 0 leptons.
- **Cutflow** and **yield tables** show how each cut reduces events; Z→νν and tt̄ dominate in the SR.
- **Control regions** (e.g. single-lepton) validate background modelling.
- The **bbdm_processor** encapsulates the full selection for reuse.

**End of the long exercise.** You have implemented a simplified bbDM search from data loading to signal region and yields. In a real analysis, next steps would include systematic uncertainties, statistical interpretation, and possibly limit setting.

### Optional: Vary the MET threshold
Re-run your cutflow and MET plot with MET > 150 GeV and MET > 250 GeV. How do the yields and shapes change? Why is the MET cut important for background rejection?